In [1]:
# CELL 1: Install
!pip -q install torchcde pyyaml scikit-learn
print("OK")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 2.3 MB/s eta 0:00:00
OK


In [2]:
# CELL 2: Imports
import os, json, pickle
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import haversine_distances

os.chdir("/kaggle/working")
print("Environment ready")


Environment ready


In [3]:
# CELL 3: Load PEMS-BAY files
H5_PATH  = "/kaggle/input/datasets/cheminichemseddine/pems-bay/pems-bay.h5"
LOCS_CSV = "/kaggle/input/datasets/cheminichemseddine/graph-sensor-locations-bay-csv/graph_sensor_locations_bay.csv"

df = pd.read_hdf(H5_PATH)
print("pems-bay.h5:", df.shape, "  (timesteps x sensors)")
print("Time range :", df.index[0], "->", df.index[-1])
print("Columns    :", list(df.columns[:5]), "... [", df.shape[1], "total ]")

sensor_ids = list(df.columns)
N = df.shape[1]   # 325
T = df.shape[0]   # ~52116

X = df.fillna(0).values.astype("float32")
print("Zero (missing):", round(100*(X==0).mean(), 2), "%")
print("Value range (non-zero): [", round(float(X[X>0].min()),2), ",", round(float(X.max()),2), "]")


pems-bay.h5: (52116, 325)   (timesteps x sensors)
Time range : 2017-01-01 00:00:00 -> 2017-06-30 23:55:00
Columns    : [400001, 400017, 400030, 400040, 400045] ... [ 325 total ]
Zero (missing): 0.0 %
Value range (non-zero): [ 3.0 , 85.1 ]


In [4]:
# CELL 4: Distance matrix + mean/std

# ── 4a. train mean / std (70% split, non-zero only) ──────
T_train   = int(T * 0.7)
X_train   = X[:T_train]

train_mean = np.array(
    [X_train[:, j][X_train[:, j] != 0].mean()
     if (X_train[:, j] != 0).any() else 0.
     for j in range(N)], dtype="float32")

train_std = np.array(
    [X_train[:, j][X_train[:, j] != 0].std()
     if (X_train[:, j] != 0).sum() > 1 else 1.
     for j in range(N)], dtype="float32")
train_std[train_std == 0] = 1.

print("train_mean: min=%.2f  max=%.2f  mean=%.2f" % (train_mean.min(), train_mean.max(), train_mean.mean()))
print("train_std : min=%.2f  max=%.2f  mean=%.2f" % (train_std.min(),  train_std.max(),  train_std.mean()))

# ── 4b. Distance matrix from GPS coordinates ─────────────
locs = pd.read_csv(LOCS_CSV, header=None, names=["sensor_id", "lat", "lon"])
print("\nLocations file:", locs.shape, "rows")
print(locs.head(3).to_string())

# align to df column order
locs = locs.set_index("sensor_id")

# sensor_ids may be int or str — coerce both to same type
try:
    locs.index = locs.index.astype(type(sensor_ids[0]))
except Exception:
    locs.index = locs.index.astype(str)
    sensor_ids_str = [str(s) for s in sensor_ids]
    locs = locs.reindex(sensor_ids_str)
else:
    locs = locs.reindex(sensor_ids)

missing_locs = locs["lat"].isna().sum()
print("\nSensors without location:", missing_locs)
if missing_locs > 0:
    locs = locs.fillna(locs.mean())

latlon     = locs[["lat", "lon"]].values
latlon_rad = np.radians(latlon)
_R = 6371.0088
dist_matrix = haversine_distances(latlon_rad) * _R   # km

print("Distance matrix:", dist_matrix.shape)
print("Range (km): [%.2f, %.2f]" % (dist_matrix[dist_matrix>0].min(), dist_matrix.max()))


train_mean: min=53.61  max=68.07  mean=62.74
train_std : min=1.01  max=17.92  mean=8.38

Locations file: (325, 3) rows
   sensor_id        lat         lon
0     400001  37.364085 -121.901149
1     400017  37.253303 -121.945440
2     400030  37.359087 -121.906538

Sensors without location: 0
Distance matrix: (325, 325)
Range (km): [0.00, 26.63]


In [5]:
# CELL 5: Stage 1 - Node metadata
print("=" * 60)
print("STAGE 1 - PER-SENSOR STATISTICS")
print("=" * 60)

ob_mask_train = (X_train != 0.)
nodes_metadata = []

for j in range(N):
    col_all   = X[:, j]
    col_train = X_train[:, j]
    obs_all   = col_all[col_all != 0.]
    obs_train = col_train[col_train != 0.]

    # missing rate
    missing_rate = float((col_all == 0.).mean())
    if   missing_rate < 0.05: missing_cat = "very_low"
    elif missing_rate < 0.10: missing_cat = "low"
    elif missing_rate < 0.20: missing_cat = "medium"
    elif missing_rate < 0.40: missing_cat = "high"
    else:                     missing_cat = "very_high"

    # signal stats (use pre-computed mean/std — same as normalisation)
    sig_mean = float(train_mean[j])
    sig_std  = float(train_std[j])

    # autocorrelation lag-1
    if len(obs_train) > 2:
        mu    = obs_train.mean()
        d     = obs_train - mu
        denom = (d ** 2).sum()
        autocorr = float((d[:-1] * d[1:]).sum() / denom) if denom > 0 else 0.
        autocorr = float(np.clip(autocorr, -1., 1.))
    else:
        autocorr = 0.

    # stability (variance of rolling variance)
    win = 48
    roll_vars = []
    for k in range(0, len(col_train) - win, win):
        chunk = col_train[k:k+win]
        chunk = chunk[chunk != 0.]
        if len(chunk) > win // 2:
            roll_vars.append(float(chunk.var()))

    if len(roll_vars) > 1:
        var_of_var = float(np.var(roll_vars))
        med_var    = float(np.median([v for v in roll_vars if v > 0])) if any(v > 0 for v in roll_vars) else 1.
        stability  = "unstable" if var_of_var > med_var * 2 else "stable"
    else:
        var_of_var = 0.
        stability  = "stable"

    nodes_metadata.append({
        "sensor_id"        : str(sensor_ids[j]),
        "sensor_index"     : j,
        "missing_rate"     : round(missing_rate, 6),
        "missing_category" : missing_cat,
        "mean"             : round(sig_mean, 4),
        "std"              : round(sig_std,  4),
        "autocorr_lag1"    : round(autocorr, 6),
        "stability"        : stability,
        "temporal_var_mean": round(float(np.mean(roll_vars)) if roll_vars else 0., 4),
        "temporal_var_std" : round(float(np.std(roll_vars))  if roll_vars else 0., 4),
    })

print("Sensors processed:", len(nodes_metadata))
print("Missing rate  : min=%.1f%%  max=%.1f%%" % (
    min(n["missing_rate"] for n in nodes_metadata)*100,
    max(n["missing_rate"] for n in nodes_metadata)*100))
print("Autocorr lag1 : mean=%.3f" % np.mean([n["autocorr_lag1"] for n in nodes_metadata]))
print("Stable sensors:", sum(n["stability"] == "stable" for n in nodes_metadata), "/", N)
print("\nSample node:")
print(json.dumps(nodes_metadata[0], indent=2))


STAGE 1 - PER-SENSOR STATISTICS
Sensors processed: 325
Missing rate  : min=0.0%  max=0.0%
Autocorr lag1 : mean=0.974
Stable sensors: 0 / 325

Sample node:
{
  "sensor_id": "400001",
  "sensor_index": 0,
  "missing_rate": 1.9e-05,
  "missing_category": "very_low",
  "mean": 67.6293,
  "std": 8.0474,
  "autocorr_lag1": 0.98217,
  "stability": "unstable",
  "temporal_var_mean": 32.4655,
  "temporal_var_std": 84.3176
}


In [6]:
# CELL 6: Stage 2 - Relation metadata
print("=" * 60)
print("STAGE 2 - PAIRWISE CORRELATION ANALYSIS")
print("=" * 60)

# geographic adjacency (same formula as PriSTI)
finite_dist = dist_matrix.reshape(-1)
finite_dist = finite_dist[~np.isinf(finite_dist)]
finite_dist = finite_dist[finite_dist > 0]
sigma       = finite_dist.std()
adj         = np.exp(-np.square(dist_matrix / sigma))
adj[adj < 0.1] = 0.
np.fill_diagonal(adj, 0.)

print("Geographic adj: sigma=%.2f km" % sigma)
print("  Non-zero entries:", np.count_nonzero(adj), "/", N*N)
print("  Weight range: [%.4f, %.4f]" % (adj[adj>0].min(), adj.max()))

MIN_CO_OBS  = 30
PEARSON_MIN = 0.15
relations_metadata = []
n_pairs   = 0
n_skipped = 0

for i in range(N):
    for j in range(i+1, N):
        gw = float(adj[i, j])
        if gw == 0.:
            continue

        mask_co = (X_train[:, i] != 0.) & (X_train[:, j] != 0.)
        n_co    = int(mask_co.sum())

        if n_co < MIN_CO_OBS:
            n_skipped += 1
            continue

        xi = X_train[mask_co, i]
        xj = X_train[mask_co, j]
        mu_i, mu_j = xi.mean(), xj.mean()
        si, sj     = xi.std(), xj.std()

        if si < 1e-8 or sj < 1e-8:
            pearson = 0.
        else:
            pearson = float(((xi - mu_i) * (xj - mu_j)).mean() / (si * sj))
            pearson = float(np.clip(pearson, -1., 1.))

        dist_km = float(dist_matrix[i, j])

        if   pearson > 0.85: rel_type = "STRONGLY_CORRELATED"
        elif pearson > 0.70: rel_type = "CORRELATED"
        else:                rel_type = "WEAKLY_CORRELATED"

        relations_metadata.append({
            "source"         : str(sensor_ids[i]),
            "target"         : str(sensor_ids[j]),
            "pearson"        : round(pearson, 6),
            "weight"         : round(pearson, 6),
            "distance_km"    : round(dist_km, 3),
            "gaussian_weight": round(gw, 6),
            "rel_type"       : rel_type,
        })
        n_pairs += 1

from collections import Counter
types    = Counter(r["rel_type"] for r in relations_metadata)
pearsons = [r["pearson"] for r in relations_metadata]
passing  = sum(1 for p in pearsons if p >= PEARSON_MIN)

print("\nRelations generated:", n_pairs, "(", n_skipped, "skipped: <", MIN_CO_OBS, "co-obs )")
print("  STRONGLY_CORRELATED (>0.85):", types["STRONGLY_CORRELATED"])
print("  CORRELATED (0.70-0.85)     :", types["CORRELATED"])
print("  WEAKLY_CORRELATED (<=0.70) :", types["WEAKLY_CORRELATED"])
print("  Pearson range: [%.3f, %.3f]  mean=%.3f" % (min(pearsons), max(pearsons), np.mean(pearsons)))
print("  Will pass delta=0.15 threshold:", passing, "->", passing*2, "directed edges after bidirectionalization")


STAGE 2 - PAIRWISE CORRELATION ANALYSIS
Geographic adj: sigma=5.36 km
  Non-zero entries: 38278 / 105625
  Weight range: [0.1000, 1.0000]

Relations generated: 19139 ( 0 skipped: < 30 co-obs )
  STRONGLY_CORRELATED (>0.85): 624
  CORRELATED (0.70-0.85)     : 2249
  WEAKLY_CORRELATED (<=0.70) : 16266
  Pearson range: [-0.144, 1.000]  mean=0.373
  Will pass delta=0.15 threshold: 13686 -> 27372 directed edges after bidirectionalization


In [7]:
# CELL 7: Save all files
OUTPUT_DIR = "/kaggle/working/pemsbay_metadata"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# nodes_metadata.json
with open(OUTPUT_DIR + "/nodes_metadata.json", "w") as f:
    json.dump(nodes_metadata, f, indent=2)
print("nodes_metadata.json       -", len(nodes_metadata), "nodes")

# relations_metadata.json
with open(OUTPUT_DIR + "/relations_metadata.json", "w") as f:
    json.dump(relations_metadata, f, indent=2)
print("relations_metadata.json   -", len(relations_metadata), "relations")

# A_static.npy
np.save(OUTPUT_DIR + "/A_static.npy", adj.astype("float32"))
print("A_static.npy              -", adj.shape, " non-zero:", np.count_nonzero(adj))

# pems_bay_dist.npy
np.save(OUTPUT_DIR + "/pems_bay_dist.npy", dist_matrix.astype("float32"))
print("pems_bay_dist.npy         -", dist_matrix.shape)

# pems_meanstd.pk
with open(OUTPUT_DIR + "/pems_meanstd.pk", "wb") as f:
    pickle.dump((train_mean, train_std), f)
print("pems_meanstd.pk           - mean/std shape=", train_mean.shape)

# var_mean / var_std (signal mean/std for anomaly gate in Layer 2)
var_mean_arr = np.array([n["mean"] for n in nodes_metadata], dtype="float32")
var_std_arr  = np.array([n["std"]  for n in nodes_metadata], dtype="float32")
np.save(OUTPUT_DIR + "/var_mean.npy", var_mean_arr)
np.save(OUTPUT_DIR + "/var_std.npy",  var_std_arr)
print("var_mean.npy / var_std.npy - shape=", var_mean_arr.shape)

print("\nAll files saved to:", OUTPUT_DIR)
print("\nWhere these files go:")
print("  data/pems_bay/        <- pems_bay.h5  pems_meanstd.pk  pems_bay_dist.npy")
print("  neo4j_setup/metadata_pemsbay/  <- nodes_metadata.json  relations_metadata.json")
print("                                    A_static.npy  var_mean.npy  var_std.npy")


nodes_metadata.json       - 325 nodes
relations_metadata.json   - 19139 relations
A_static.npy              - (325, 325)  non-zero: 38278
pems_bay_dist.npy         - (325, 325)
pems_meanstd.pk           - mean/std shape= (325,)
var_mean.npy / var_std.npy - shape= (325,)

All files saved to: /kaggle/working/pemsbay_metadata

Where these files go:
  data/pems_bay/        <- pems_bay.h5  pems_meanstd.pk  pems_bay_dist.npy
  neo4j_setup/metadata_pemsbay/  <- nodes_metadata.json  relations_metadata.json
                                    A_static.npy  var_mean.npy  var_std.npy


In [8]:
# CELL 8: Validation
print("=" * 60)
print("VALIDATION")
print("=" * 60)

print("\nA_static:")
print("  Shape      :", adj.shape)
print("  Symmetric  :", np.allclose(adj, adj.T))
print("  Diagonal=0 :", bool(np.all(np.diag(adj) == 0)))
print("  Non-zero   :", np.count_nonzero(adj))

print("\nW_sem preview (what Layer 1 will build):")
n_pass = sum(1 for r in relations_metadata if r["pearson"] >= 0.15)
print("  Pairs passing delta=0.15 :", n_pass)
print("  Directed edges (x2)      :", n_pass * 2)
neg = [r["pearson"] for r in relations_metadata if r["pearson"] < 0]
print("  Negative Pearson edges   :", len(neg), "(will be filtered)")

print("\nAnomaly thresholds - sample sensors:")
for i in range(3):
    n = nodes_metadata[i]
    lo = round(n["mean"] - 3*n["std"], 1)
    hi = round(n["mean"] + 3*n["std"], 1)
    print("  Sensor", i, ": mean=%.1f  std=%.1f  +-3sigma=[%.1f, %.1f]" % (n["mean"], n["std"], lo, hi))

print("\nValidation complete - PEMS-BAY metadata ready")


VALIDATION

A_static:
  Shape      : (325, 325)
  Symmetric  : True
  Diagonal=0 : True
  Non-zero   : 38278

W_sem preview (what Layer 1 will build):
  Pairs passing delta=0.15 : 13686
  Directed edges (x2)      : 27372
  Negative Pearson edges   : 1753 (will be filtered)

Anomaly thresholds - sample sensors:
  Sensor 0 : mean=67.6  std=8.0  +-3sigma=[43.5, 91.8]
  Sensor 1 : mean=59.2  std=13.1  +-3sigma=[19.9, 98.4]
  Sensor 2 : mean=59.2  std=11.7  +-3sigma=[24.0, 94.3]

Validation complete - PEMS-BAY metadata ready


In [9]:
# CELL 9: Download
import shutil
shutil.make_archive("/kaggle/working/pemsbay_metadata_all", "zip", OUTPUT_DIR)
print("Download: /kaggle/working/pemsbay_metadata_all.zip")
print("\nContents:")
for fname in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(OUTPUT_DIR + "/" + fname)
    print("  %-35s %.1f KB" % (fname, size/1024))


Download: /kaggle/working/pemsbay_metadata_all.zip

Contents:
  A_static.npy                        412.7 KB
  nodes_metadata.json                 92.3 KB
  pems_bay_dist.npy                   412.7 KB
  pems_meanstd.pk                     2.7 KB
  relations_metadata.json             3737.8 KB
  var_mean.npy                        1.4 KB
  var_std.npy                         1.4 KB
